# Pico-models — T4 GPU Training

Optimised for NVIDIA T4 (15 GB VRAM). Run cells top-to-bottom.

| Model | Task | VRAM | Batch |
|-------|------|------|-------|
| PicoLLM | Text generation | ~2 GB | 64 |
| PicoVAE | Image generation | ~3 GB | 64 |
| PicoVLM | Vision + Language | ~6 GB | 16 |
| PicoVLM + grad-ckpt | Vision + Language | ~4 GB | 24 |

Precision: T4 uses `fp16` automatically (Turing cc 7.5).

## 1. Setup

In [ ]:
!unzip -q pico-models.zip
%cd pico-models
!pip install -q torch torchvision Pillow sentencepiece datasets

In [ ]:
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cc   = torch.cuda.get_device_capability()
    print(f'GPU: {name}  ({vram:.1f} GB)  cc={cc}')
    if cc[0] < 8:
        print('Turing/Volta detected — fp16 mode (optimal for T4/V100)')
    else:
        print('Ampere+ detected — bf16 mode')
else:
    print('WARNING: No GPU found')

## 2. Prepare Data

In [ ]:
import os
os.makedirs('data/images', exist_ok=True)

# Download CIFAR-10 and save 1000 images to data/images/
from torchvision.datasets import CIFAR10
ds = CIFAR10('.', download=True)
for i, (img, _) in enumerate(ds):
    if i >= 1000:
        break
    img.save(f'data/images/{i}.png')

print(f'Saved {min(1000, len(ds))} images to data/images/')

In [ ]:
# Build corpus.txt from fineweb-edu (streaming — stops after 50k chars)
from datasets import load_dataset

MAX_CHARS = 5_000_000  # ~5 MB, enough for a quick training run

ds = load_dataset('HuggingFaceFW/fineweb-edu', 'default', split='train', streaming=True)
chars = 0
with open('data/corpus.txt', 'w') as f:
    for row in ds:
        text = row['text'] + '\n'
        f.write(text)
        chars += len(text)
        if chars >= MAX_CHARS:
            break

print(f'corpus.txt: {chars/1e6:.1f} MB')

In [ ]:
# Build chats.jsonl from a real instruction dataset (SlimOrca subset)
# Format: {prompt, thinking, response} — thinking field is optional
from datasets import load_dataset
import json, re

N_SAMPLES = 5000   # increase for better quality, decrease if slow

ds = load_dataset('Open-Orca/SlimOrca', split='train', streaming=True)

written = 0
with open('data/chats.jsonl', 'w') as f:
    for row in ds:
        convs = row.get('conversations', [])
        # SlimOrca format: [{from: system/human/gpt, value: ...}, ...]
        human = next((c['value'] for c in convs if c['from'] == 'human'), None)
        gpt   = next((c['value'] for c in convs if c['from'] == 'gpt'),   None)
        if not human or not gpt:
            continue
        # Keep short samples so char tokenizer vocab stays manageable
        if len(human) + len(gpt) > 800:
            continue
        entry = {'prompt': human.strip(), 'response': gpt.strip()}
        f.write(json.dumps(entry) + '\n')
        written += 1
        if written >= N_SAMPLES:
            break

print(f'data/chats.jsonl: {written} samples')


In [ ]:
# Build vlm_data.jsonl from the CIFAR images + captions
import json, os, random

CIFAR_LABELS = ['airplane','automobile','bird','cat','deer',
                'dog','frog','horse','ship','truck']

from torchvision.datasets import CIFAR10
ds = CIFAR10('.', download=False)

with open('data/vlm_data.jsonl', 'w') as f:
    for i in range(1000):
        label = CIFAR_LABELS[ds.targets[i]]
        path  = os.path.abspath(f'data/images/{i}.png')
        entry = {
            'image':    path,
            'prompt':   'What is in this image?',
            'thinking': f'I look at the image carefully. It appears to show a {label}.',
            'response': f'This is a {label}.'
        }
        f.write(json.dumps(entry) + '\n')

print('data/vlm_data.jsonl: 1000 image-caption pairs')

In [ ]:
# Verify all data files exist
import os
files = [
    ('data/corpus.txt',    'LLM text corpus'),
    ('data/chats.jsonl',   'Chat + thinking pairs'),
    ('data/vlm_data.jsonl','VLM image-caption pairs'),
]
imgs = len([f for f in os.listdir('data/images') if f.endswith('.png')])
print(f'data/images/: {imgs} images')
for path, desc in files:
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e6
        print(f'{path}: {size:.2f} MB  ({desc})')
    else:
        print(f'MISSING: {path}')

## 3. LLM Training (Text)

In [ ]:
# LLM — plain text corpus
# 15M param model, vocab=4096, BPE-style char vocab
# T4 15GB: batch=32, seq=512, fp16 auto, ~4 GB VRAM

!python train.py train-llm \
  --data data/corpus.txt \
  --out  checkpoints/llm \
  --vocab-size 4096 \
  --dim 512 --layers 8 --heads 8 --kv-heads 2 \
  --seq-len 512 \
  --batch 32 --accum 4 \
  --steps 10000 \
  --log-every 100 --save-every 2000


In [ ]:
# LLM — chat with thinking (SlimOrca data)
# ~15M params, vocab=4096 (all unique chars in dataset)
# T4 15GB: batch=16, seq=512, ~4 GB VRAM
#
# Why this works much better than synthetic data:
#   - 5000 diverse real conversations vs 2000 repeated math templates
#   - vocab=4096 captures punctuation/words properly
#   - dim=512 gives the model capacity to learn patterns

!python train.py train-llm \
  --data data/chats.jsonl \
  --out  checkpoints/llm-chat \
  --vocab-size 4096 \
  --dim 512 --layers 8 --heads 8 --kv-heads 2 \
  --seq-len 512 \
  --batch 16 --accum 8 \
  --steps 10000 \
  --log-every 100 --save-every 2000


In [ ]:
# Quick sanity check — run after at least 2000 steps
# Should see coherent words/sentences, not random letters
import torch, sys
sys.path.insert(0, '.')
from pico.arch import PicoConfig, PicoLLM
from pico.tokenizer import CharTokenizer

ckpt  = torch.load('checkpoints/llm-chat/ckpt_final.pt', map_location='cuda', weights_only=False)
cfg   = PicoConfig(**ckpt['config'])
model = PicoLLM(cfg).cuda().eval()
model.load_state_dict(ckpt['model'])
tok   = CharTokenizer.load('checkpoints/llm-chat/tokenizer.json')

print(f'Model: {model.num_params/1e6:.1f}M params | vocab={cfg.vocab_size}')

for test_prompt in ['What is gravity?', 'How do I cook rice?', 'Tell me a joke.']:
    formatted = f'<|user|>{test_prompt}<|assistant|>'
    ids    = tok.encode(formatted, bos=True, eos=False)
    tokens = torch.tensor([ids], device='cuda')
    with torch.no_grad():
        out = model.generate(tokens, max_new_tokens=120, temperature=0.8,
                             top_p=0.9, stop_ids=[tok.EOS])
    print(f'Q: {test_prompt}')
    print(f'A: {tok.decode(out[0, len(ids):].tolist())}')
    print()


## 4. Image Generation Training (VAE)

In [ ]:
# VAE — 128x128 images, batch=64, ~3 GB VRAM

!python train.py train-img \
  --data data/images/ \
  --out  checkpoints/vae \
  --img-size 128 --patch-size 16 --latent-dim 64 \
  --dim 256 --layers 4 --heads 8 \
  --batch 64 --accum 1 \
  --steps 5000 \
  --log-every 50 --save-every 1000

In [ ]:
# VAE — higher quality 256x256, ~5 GB VRAM

!python train.py train-img \
  --data data/images/ \
  --out  checkpoints/vae-256 \
  --img-size 256 --patch-size 16 --latent-dim 128 \
  --dim 384 --layers 6 --heads 8 \
  --batch 32 --accum 2 \
  --steps 8000 \
  --log-every 50 --save-every 1000

## 5. Vision-Language Model (VLM) Training

In [ ]:
# VLM — 128x128 images, batch=16, ~6 GB VRAM
# Visual tokens: (128/16)^2 + 1 = 65 tokens prepended per sample

!python train.py train-vlm \
  --data data/vlm_data.jsonl \
  --out  checkpoints/vlm \
  --img-size 128 --patch-size 16 \
  --dim 256 --layers 6 --heads 8 --kv-heads 2 \
  --seq-len 256 \
  --batch 16 --accum 4 \
  --steps 5000 \
  --log-every 50 --save-every 1000

In [ ]:
# VLM — memory efficient with gradient checkpointing (~4 GB VRAM)
# --grad-ckpt: saves ~40% VRAM, costs ~15% speed
# Allows larger batch: 24 instead of 16

!python train.py train-vlm \
  --data data/vlm_data.jsonl \
  --out  checkpoints/vlm-ckpt \
  --img-size 128 --patch-size 16 \
  --dim 256 --layers 6 --heads 8 --kv-heads 2 \
  --seq-len 256 \
  --batch 24 --accum 4 --grad-ckpt \
  --steps 5000 \
  --log-every 50 --save-every 1000

In [ ]:
# VLM — with thinking traces
# Requires vlm_data.jsonl with 'thinking' field (already built in section 2)

!python train.py train-vlm \
  --data data/vlm_data.jsonl \
  --out  checkpoints/vlm-think \
  --img-size 128 --patch-size 16 \
  --dim 256 --layers 6 --heads 8 --kv-heads 2 \
  --seq-len 512 \
  --batch 16 --accum 4 --grad-ckpt \
  --steps 5000 --thinking \
  --log-every 50 --save-every 1000

## 6. Inference

In [ ]:
# Benchmark: params + tok/s
!python run.py bench \
  --ckpt checkpoints/llm/ckpt_final.pt \
  --tok  checkpoints/llm/tokenizer.json

In [ ]:
# LLM generation
import torch, sys
sys.path.insert(0, '.')
from pico.arch import PicoConfig, PicoLLM
from pico.tokenizer import CharTokenizer

ckpt  = torch.load('checkpoints/llm/ckpt_final.pt', map_location='cuda', weights_only=False)
cfg   = PicoConfig(**ckpt['config'])
model = PicoLLM(cfg).cuda().eval()
model.load_state_dict(ckpt['model'])
tok   = CharTokenizer.load('checkpoints/llm/tokenizer.json')

prompt = 'The study of'
ids    = tok.encode(prompt, bos=True, eos=False)
tokens = torch.tensor([ids], device='cuda')

with torch.no_grad():
    out = model.generate(tokens, max_new_tokens=200, temperature=0.8, top_p=0.9, stop_ids=[tok.EOS])

print(prompt + tok.decode(out[0, len(ids):].tolist()))

In [ ]:
# LLM with thinking — shows reasoning trace + final answer
import torch, sys
sys.path.insert(0, '.')
from pico.arch import PicoConfig, PicoLLM
from pico.tokenizer import CharTokenizer

ckpt  = torch.load('checkpoints/llm-think/ckpt_final.pt', map_location='cuda', weights_only=False)
cfg   = PicoConfig(**ckpt['config'])
model = PicoLLM(cfg).cuda().eval()
model.load_state_dict(ckpt['model'])
tok   = CharTokenizer.load('checkpoints/llm-think/tokenizer.json')

prompt = '<|user|>What is 37 + 48?<|assistant|>'
ids    = tok.encode(prompt, bos=True, eos=False)
tokens = torch.tensor([ids], device='cuda')

with torch.no_grad():
    out = model.generate(tokens, max_new_tokens=200, temperature=0.7, stop_ids=[tok.EOS])

raw = out[0, len(ids):].tolist()
answer, thinking = tok.decode_with_thinking(raw)
if thinking:
    print(f'[Thinking] {thinking}')
print(f'[Answer]   {answer}')

In [ ]:
# VAE — generate random images
import torch, sys
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
sys.path.insert(0, '.')
from pico.arch import PicoConfig, PicoVAE

ckpt  = torch.load('checkpoints/vae/ckpt_vae_final.pt', map_location='cuda', weights_only=False)
cfg   = PicoConfig(**ckpt['config'])
model = PicoVAE(cfg).cuda().eval()
model.load_state_dict(ckpt['model'])

with torch.no_grad():
    samples = model.sample(8, device='cuda')

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(TF.to_pil_image(samples[i].cpu().clamp(0, 1)))
    ax.axis('off')
plt.suptitle('PicoVAE — Generated Images')
plt.tight_layout()
plt.savefig('generated_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# VLM — describe an image
import torch, sys
from PIL import Image
import torchvision.transforms as T
sys.path.insert(0, '.')
from pico.arch import PicoConfig, PicoVLM
from pico.tokenizer import CharTokenizer

ckpt  = torch.load('checkpoints/vlm/ckpt_vlm_final.pt', map_location='cuda', weights_only=False)
cfg   = PicoConfig(**ckpt['config'])
model = PicoVLM(cfg).cuda().eval()
model.load_state_dict(ckpt['model'])
tok   = CharTokenizer.load('checkpoints/vlm/tokenizer.json')

img_path = 'data/images/0.png'
img = Image.open(img_path).convert('RGB')
tfm = T.Compose([
    T.Resize((cfg.img_size, cfg.img_size)),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])
img_tensor = tfm(img).unsqueeze(0).cuda()

prompt = '<|user|>What is in this image?<|assistant|>'
ids    = tok.encode(prompt, bos=True, eos=False)
tokens = torch.tensor([ids], device='cuda')

with torch.no_grad():
    out = model.generate(tokens, images=img_tensor, max_new_tokens=100,
                         temperature=0.7, stop_ids=[tok.EOS])

print('Response:', tok.decode(out[0, len(ids):].tolist()))

import matplotlib.pyplot as plt
plt.imshow(img); plt.axis('off'); plt.title('Input image'); plt.show()